# What else does embpy know about your entities?

**What you'll learn.** An embedding tells you where an entity sits relative to
others. It does not tell you what the entity *is*. embpy's annotation layer
fills that in from public databases — physicochemical properties, clinical
history, pathways, targets, disease links — and writes it into the same AnnData
next to the vectors.

That matters for two reasons beyond curiosity:

- **An unlabelled embedding cannot be evaluated.** Annotations give you labels
  nobody in your analysis chose, which is what turns "the clusters look
  sensible" into a number.
- **Some facts are not in the data.** No structural model can tell you a drug
  was withdrawn for cardiotoxicity in 2004. That is a record, not a signal.

| Section | Question |
| --- | --- |
| [Molecules](#molecules-structure-first) | what is this compound, physically? |
| [The clinical record](#the-clinical-record) | is it a drug, what for, and is it safe? |
| [Genes](#genes) | what pathways and diseases is this gene in? |
| [Proteins](#proteins) | what does this protein do, and where? |
| [Putting them to work](#putting-annotations-to-work) | how do annotations score an embedding? |

**Prerequisites.** [Embed with any model](01_embed_any_model.ipynb) and
[Where embeddings live](02_output_contract.ipynb).

## Requirements

The annotation layer is **network-bound, not compute-bound**. There is no model
to download and no GPU involved; every function here queries a public API, so
what it needs is egress and patience.

The core install covers everything in this notebook:

```bash
uv pip install "embpy[cpu]"
```

Two functions reach for optional extras:

| Function | Needs | Install |
| --- | --- | --- |
| `tl.annotate_molecules` | RDKit (already in core) | — |
| `tl.annotate_drug_perturbations` | nothing beyond core | — |
| `tl.annotate_gene_perturbations` | nothing beyond core | — |
| `tl.annotate_proteins` | nothing beyond core | — |
| `tl.annotate_drugs`, `annotate_cell_lines`, `annotate_drug_response` | pertpy | `uv pip install "embpy[pertpy]"` |

Rough cost, so the slow cells are not a surprise: the ChEMBL record is about a
dozen requests per compound, and the gene annotator queries seven separate
services. Both are polite by default (a small delay between calls), so a
forty-compound panel takes minutes rather than seconds.

In [1]:
import os

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import anndata as ad
import numpy as np
import pandas as pd
from IPython.display import display

from embpy import tl

## Molecules: structure first

`tl.annotate_molecules` is the local half — RDKit only, no network. It reads
identifiers from `.obs[column]` and writes `mol_*` columns: molecular weight,
LogP, TPSA, hydrogen-bond donors and acceptors, rotatable bonds, QED, Lipinski
violations, Fsp3 and heavy-atom count.

Note the identifiers below are *names*, not SMILES. embpy resolves them.

In [2]:
DRUGS = ["aspirin", "imatinib", "rofecoxib", "atorvastatin", "diazepam"]

mols = ad.AnnData(
    X=np.zeros((len(DRUGS), 1), dtype=np.float32),
    obs=pd.DataFrame({"compound": DRUGS}, index=DRUGS),
)
mols = tl.annotate_molecules(mols, column="compound", sources=["structural"])

mol_cols = [c for c in mols.obs.columns if c.startswith("mol_")]
print(f"{len(mol_cols)} mol_* columns")
display(mols.obs[["mol_molecular_weight", "mol_logp", "mol_tpsa",
                  "mol_qed", "mol_lipinski_violations"]].round(2))

12 mol_* columns


,mol_molecular_weight,mol_logp,mol_tpsa,mol_qed,mol_lipinski_violations
aspirin,180.16,1.31,63.60,0.55,0
imatinib,493.62,4.59,86.28,0.39,0
rofecoxib,314.36,2.56,60.44,0.82,0
atorvastatin,558.65,6.31,111.79,0.16,2
diazepam,284.75,3.15,32.67,0.79,0


`sources=["structural"]` keeps this cell offline-fast. Dropping it queries ChEBI
roles, KEGG pathways, PubChem cross-references and disease associations too —
useful, but each is a network round trip per compound.

## The clinical record

`tl.annotate_drug_perturbations` is the ChEMBL half, and it answers the
questions a structure cannot: what phase did this compound reach, what is it
given for, was it withdrawn and why, where does the WHO file it, what does it
inhibit, and how is it cleared.

It writes 38 `drug_*` columns. The interesting ones are below; the complete
records — every indication, every warning with its references, the full target
profile — land in `.uns["chembl_annotations"]`.

In [3]:
mols = tl.annotate_drug_perturbations(mols, column="compound", sources="all")

drug_cols = [c for c in mols.obs.columns if c.startswith("drug_")]
print(f"{len(drug_cols)} drug_* columns, "
      f"{int(mols.obs['drug_in_chembl'].sum())}/{mols.n_obs} matched in ChEMBL")

display(mols.obs[[
    "drug_pref_name", "drug_development_phase", "drug_first_approval",
    "drug_atc_level1", "drug_moa",
]])

39 drug_* columns, 5/5 matched in ChEMBL


,drug_pref_name,drug_development_phase,drug_first_approval,drug_atc_level1,drug_moa
aspirin,ASPIRIN,approved,1950,BLOOD AND BLOOD FORMING ORGANS,Cyclooxygenase inhibitor
imatinib,IMATINIB,approved,2001,ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS,Tyrosine-protein kinase ABL inhibitor
rofecoxib,ROFECOXIB,approved,1999,MUSCULO-SKELETAL SYSTEM,Cyclooxygenase-2 inhibitor
atorvastatin,ATORVASTATIN,approved,1996,CARDIOVASCULAR SYSTEM,HMG-CoA reductase inhibitor
diazepam,DIAZEPAM,approved,1963,NERVOUS SYSTEM,GABA-A receptor; anion channel positive allost...


### The part no model can infer

Two of these compounds are chemically unremarkable and clinically very
different. Rofecoxib and celecoxib inhibit the same enzyme; one was withdrawn
worldwide and the other was not. That difference lives in a database, not in a
molecule.

In [4]:
display(mols.obs[[
    "drug_is_withdrawn", "drug_has_black_box_warning",
    "drug_n_warnings", "drug_withdrawn_reason",
]])

# The obs columns are a summary; the full record keeps the class, country,
# year and the references behind each warning.
for warning in (mols.uns["chembl_annotations"]["rofecoxib"]["warnings"] or [])[:3]:
    print(f"  {warning['warning_type']:20s} {str(warning['warning_class']):16s} "
          f"{str(warning['warning_country']):12s} {warning['warning_year']}")

,drug_is_withdrawn,drug_has_black_box_warning,drug_n_warnings,drug_withdrawn_reason
aspirin,False,False,0,
imatinib,False,False,0,
rofecoxib,True,True,11,cardiotoxicity
atorvastatin,False,False,0,
diazepam,False,True,2,


  Black Box Warning    None             United States None
  Withdrawn            cardiotoxicity   Worldwide    2004
  Withdrawn            cardiotoxicity   Worldwide    2004


### Targets as genes, not accessions

A `CHEMBL1862` is not an answer to "what does this drug hit". The target
profile resolves each target to a gene symbol, a UniProt accession and a
protein family, and summarises the measurements behind it.

In [5]:
from embpy.resources import ChEMBLAnnotator

chembl = ChEMBLAnnotator(rate_limit_delay=0.1)
profile = chembl.get_target_profile("imatinib", limit=150)

display(pd.DataFrame([
    {"target": (e["target_name"] or "")[:32], "type": e.get("target_type"),
     "gene": e.get("gene_symbol"), "class": e.get("target_class"),
     "n": e["n_measurements"], "best": e["best_pchembl"],
     "median": e["median_pchembl"]}
    for e in profile[:6]
]).set_index("target").round(2))

,type,gene,class,n,best,median
target,,,,,,
Receptor tyrosine-protein kinase,SINGLE PROTEIN,ERBB2,None,1,10.22,10.22
Epidermal growth factor receptor,SINGLE PROTEIN,EGFR,None,1,9.96,9.96
K562,CELL-LINE,None,None,7,9.82,7.31
EOL1,CELL-LINE,None,None,2,9.70,9.70
Epithelial discoidin domain-cont,SINGLE PROTEIN,DDR1,None,5,9.15,9.10
Tyrosine-protein kinase ABL1,SINGLE PROTEIN,ABL1,None,33,9.00,7.62


> **Read `n` before believing `best`.** A ChEMBL "target" can be a cell line
> rather than a protein, and a single optimistic assay outranks a target with
> dozens of consistent measurements. `summarize_selectivity` filters the first
> problem and takes `min_measurements` / `rank_by` for the second — the answer
> genuinely changes with those settings, so pick them deliberately.

In [6]:
for label, kwargs in [
    ("default", {}),
    ("min_measurements=5", {"min_measurements": 5}),
    ("min 5, by median", {"min_measurements": 5, "rank_by": "median_pchembl"}),
]:
    s = chembl.summarize_selectivity(profile, **kwargs)
    print(f"  {label:20s} -> {str(s['primary_target_gene']):8s} "
          f"n={s['primary_target_n_measurements']}")

  default              -> ERBB2    n=1
  min_measurements=5   -> DDR1     n=5
  min 5, by median     -> DDR1     n=5


## Genes

`tl.annotate_gene_perturbations` queries seven services — MyGene for pathways,
GTEx for tissue expression, HPA for localisation, STRING for interaction
partners, DoRothEA for transcription factors, Open Targets for disease links and
the GWAS Catalog for associations — and writes `gene_*` summary columns.

In [7]:
GENES = ["TP53", "EGFR", "MYC"]

genes = ad.AnnData(
    X=np.zeros((len(GENES), 1), dtype=np.float32),
    obs=pd.DataFrame({"symbol": GENES}, index=GENES),
)
genes = tl.annotate_gene_perturbations(genes, column="symbol",
                                       sources=["pathways", "interactions"])

display(genes.obs[[c for c in genes.obs.columns if c.startswith("gene_")]])

,gene_n_pathways,gene_n_ppi_partners,gene_n_disease_assoc,gene_n_transcription_factors,gene_top_tissue,gene_n_ppi_partners_at_limit,gene_n_disease_assoc_at_limit
TP53,108,10,0,0,,True,False
EGFR,84,10,0,0,,True,False
MYC,96,10,0,0,,True,False


> **A capped count is not a measurement.** `gene_n_ppi_partners` counts what was
> *fetched*, and the fetch is capped, so for a well-studied gene it reports the
> cap rather than the truth. The `*_at_limit` companion columns say which rows
> are saturated, and `.uns["gene_annotation_limits"]` records the caps.
> Regressing on a capped count without checking the flag is a silent mistake.

In [8]:
limits = genes.uns.get("gene_annotation_limits", {})
print("caps:", dict(limits))
display(genes.obs[[c for c in genes.obs.columns if c.endswith("_at_limit")]])

caps: {'gene_n_ppi_partners': 10, 'gene_n_disease_assoc': 20}


,gene_n_ppi_partners_at_limit,gene_n_disease_assoc_at_limit
TP53,True,False
EGFR,True,False
MYC,True,False


## Proteins

`tl.annotate_proteins` reads UniProt and InterPro: what the protein does, where
it sits, its domains and PTMs, disease involvement, interaction cross-references,
isoforms, and whether the entry is reviewed (Swiss-Prot) or not (TrEMBL).

In [9]:
prots = ad.AnnData(
    X=np.zeros((len(GENES), 1), dtype=np.float32),
    obs=pd.DataFrame({"symbol": GENES}, index=GENES),
)
prots = tl.annotate_proteins(prots, column="symbol",
                             sources=["function", "domains"])

display(prots.obs[[c for c in prots.obs.columns if c.startswith("prot_")]])

,prot_reviewed,prot_name,prot_n_domains,prot_n_ptms,prot_n_diseases,prot_n_interactions,prot_n_isoforms,prot_location
TP53,False,,0,0,0,0,0,
EGFR,False,,0,0,0,0,0,
MYC,False,,0,0,0,0,0,


`prot_reviewed` is worth a glance before trusting the rest: an unreviewed TrEMBL
entry is a computational prediction, and its domain and PTM counts carry much
less weight than a curated Swiss-Prot record's.

## Putting annotations to work

The payoff is not the table — it is that an annotation is a label your analysis
did not choose. Embed the compounds, then score whether the embedding recovers a
label that came from ChEMBL rather than from you.

In [10]:
from embpy import BioEmbedder, pl

embedder = BioEmbedder(device="auto", organism="human")

mols.obs["smiles"] = mols.obs["mol_canonical_smiles"]
mols.obs_names = pd.Index(mols.obs["smiles"], name="canonical_smiles")

mols = embedder.embed(
    mols, entity_type="molecule", id_type="smiles", obs_column="smiles",
    model="morgan_fp", output="anndata", attach_to="obs", key="X_morgan",
)
print("embedding:", mols.obsm["X_morgan"].shape)

embedding: (5, 2048)


In [11]:
# A label with at least two populated classes is the minimum for any purity
# score; with five compounds this is a demonstration of the mechanism, not a
# result to quote.
label = "drug_atc_level1"
usable = mols.obs[label].astype("string")
print(f"{label}: {usable.nunique()} distinct classes over {mols.n_obs} compounds")
display(usable.value_counts().to_frame("n_compounds"))

drug_atc_level1: 5 distinct classes over 5 compounds


,n_compounds
drug_atc_level1,
BLOOD AND BLOOD FORMING ORGANS,1
ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS,1
MUSCULO-SKELETAL SYSTEM,1
CARDIOVASCULAR SYSTEM,1
NERVOUS SYSTEM,1


With a real panel you would hand that column to `pl.knn_label_purity` or
`tl.compute_scib_metrics` and get a score per model. The
[small molecules](small_molecules.ipynb) notebook does exactly that across eight
embedding spaces and three annotation-derived labels — and finds, unsurprisingly,
that a structural fingerprint recovers a therapeutic classification only as far
as therapy correlates with scaffold.

## What is stored where

| Where | What |
| --- | --- |
| `.obs["mol_*"]` | physicochemical scalars |
| `.obs["drug_*"]` | the ChEMBL clinical summary, 38 columns |
| `.obs["gene_*"]`, `.obs["prot_*"]` | gene and protein summaries |
| `.obs["*_at_limit"]` | which counts hit a fetch cap |
| `.uns["molecule_annotations"]` | full per-molecule records |
| `.uns["chembl_annotations"]` | full ChEMBL records — every indication, warning, mechanism, target |
| `.uns["chembl_annotation_limits"]` | the caps themselves |

The summary columns are for filtering and colouring; the `.uns` records are for
reading. Anything the summary flattens away — a warning's references, an
indication's per-indication phase — is in `.uns`.

## Takeaway

- Annotation is **network-bound**: no weights, no GPU, just APIs and patience.
- Summary columns land in `.obs`, full records in `.uns`, and capped counts are
  flagged rather than left to look like measurements.
- The reason to bother is evaluation. An annotation is a label you did not
  choose, and that is the only kind worth scoring an embedding against.

**Next:** [Comparing embeddings](03_compare_embeddings.ipynb) for the metrics,
or [Small molecules](small_molecules.ipynb) for the same annotation layer used in
anger across a forty-compound panel.